# Ultimate OCR Pipeline (Champion Omega2)

Этот ноутбук воспроизводит финальный pipeline и добавляет Kaggle bootstrap для зависимостей.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()
print('IS_KAGGLE =', IS_KAGGLE)

if IS_KAGGLE:
    req_candidates = [
        Path('/kaggle/working/final/07_requirements_kaggle.txt'),
        Path('07_requirements_kaggle.txt'),
        Path('final/07_requirements_kaggle.txt'),
    ]

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'pip'])
    subprocess.check_call(['apt-get', 'update'])
    subprocess.check_call(['apt-get', 'install', '-y', 'libzbar0'])

    req_file = next((p for p in req_candidates if p.exists()), None)
    if req_file is not None:
        print('Installing from requirements:', req_file)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(req_file)])
    else:
        pkgs = [
            'numpy==1.26.4', 'pandas==2.2.3', 'matplotlib==3.8.4', 'tqdm==4.67.1',
            'scipy==1.12.0', 'scikit-image==0.22.0', 'scikit-learn==1.4.2',
            'opencv-python==4.10.0.84', 'opencv-contrib-python==4.10.0.84',
            'pillow==10.4.0', 'pyzbar==0.1.9',
            'paddlepaddle==3.2.0', 'paddleocr==3.2.0', 'gdown==5.2.0',
        ]
        print('Installing inline pinned dependencies...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pkgs])

    print('Bootstrap done. Рекомендуется Restart Session и затем Run all.')
else:
    print('Локальный режим: зависимости ставятся отдельно из final/08_requirements_local.txt')


## Конфигурация путей

При необходимости поменяйте `PROJECT_ROOT` под вашу среду.

In [ ]:
from pathlib import Path
import os

if Path('/kaggle').exists():
    PROJECT_ROOT = Path('/kaggle/working/LentaHack26')
else:
    PROJECT_ROOT = Path('/Users/ilya-kolosov/Developer/LentaHack26')

DATASET_ROOT = PROJECT_ROOT / 'top_crops'
TASK_PATH = PROJECT_ROOT / 'lenta_tech_life_hack_text.md'
NOTEBOOK_PATH = PROJECT_ROOT / 'notebookc9d692d630.ipynb'
OUTPUT_ROOT = PROJECT_ROOT / 'remote_outputs' / 'final_repro_omega2'
PRODUCTS_DICT = PROJECT_ROOT / 'products_v2_merged.csv'
GOOGLE_DICT = PROJECT_ROOT / 'google_dict_normalized.csv'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASET_ROOT exists =', DATASET_ROOT.exists())
print('TASK_PATH exists =', TASK_PATH.exists())
print('NOTEBOOK_PATH exists =', NOTEBOOK_PATH.exists())
print('PRODUCTS_DICT exists =', PRODUCTS_DICT.exists())
print('GOOGLE_DICT exists =', GOOGLE_DICT.exists())


## Опционально: докачать Google словарь (если отсутствует)

In [ ]:
import subprocess
import sys
import pandas as pd
from pathlib import Path

if not GOOGLE_DICT.exists() and Path('/kaggle').exists():
    file_id = '1xkYv8yRTF-jTMKYxOqKGFcwbTQuBoMYL'
    raw_payload = PROJECT_ROOT / 'google_drive_payload'
    print('Downloading Google dictionary payload...')
    subprocess.check_call([sys.executable, '-m', 'gdown', '--id', file_id, '-O', str(raw_payload)])

    df = pd.read_csv(raw_payload, sep=';', encoding='cp1251')
    df.to_csv(GOOGLE_DICT, index=False)
    print('Saved normalized CSV:', GOOGLE_DICT)
else:
    print('Skip: dictionary already exists or not Kaggle environment.')


## Запуск champion bundle (`omega2_fixed`)

In [ ]:
import subprocess
import sys

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

bundle_spec = 'omega2_fixed=data_input/H2,preprocess/H1,ocr/H4,parsers/H2,parsers/H4,qr_barcode/H1,track_merge/H1'

cmd = [
    sys.executable, 'hypothesis_campaign.py', 'run_bundle',
    '--project-root', '.',
    '--dataset-root', './top_crops',
    '--task-path', './lenta_tech_life_hack_text.md',
    '--notebook', 'notebookc9d692d630.ipynb',
    '--output-root', str(OUTPUT_ROOT),
    '--mode', 'sample',
    '--sample-size', '96',
    '--visual-panel-size', '24',
    '--seed', '123',
    '--timeout', '-1',
    '--jupyter-cmd', 'jupyter',
    '--products-dict-csv', './products_v2_merged.csv',
    '--from-scratch',
    '--guardrail-tolerance-pp', '1.0',
    '--bundle', bundle_spec,
]

if GOOGLE_DICT.exists():
    cmd.extend(['--google-dict-csv', './google_dict_normalized.csv'])

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)


## Считать итоговые метрики

In [ ]:
import json
import pandas as pd
from pathlib import Path

table_path = OUTPUT_ROOT / 'candidate_table.csv'
assert table_path.exists(), f'candidate table not found: {table_path}'

df = pd.read_csv(table_path)
display(df.tail(5))

last = df.iloc[-1]
run_id = str(last['run_id'])
run_dir = OUTPUT_ROOT / run_id
metrics_path = run_dir / 'metrics_v2.json'

metrics = json.loads(metrics_path.read_text())
print('run_id =', run_id)
print('rows =', metrics.get('rows'))
print('proxy_score =', metrics.get('proxy_score'))
print('case_proxy_v2 =', metrics.get('case_proxy_v2'))
print('price_any_fill =', metrics.get('price_any_fill'))
print('product_name_fill =', (metrics.get('fill_rate', {}) or {}).get('product_name'))
print('barcode_fill =', (metrics.get('fill_rate', {}) or {}).get('barcode'))


## Ручная проверка: выборка треков

In [ ]:
import pandas as pd

result_csv = run_dir / 'outputs_ocr_baseline' / 'result.csv'
out_df = pd.read_csv(result_csv)

cols = ['filename', 'product_name', 'price_default', 'price_discount', 'price_card', 'barcode']
print('result rows =', len(out_df))
display(out_df[cols].sample(min(12, len(out_df)), random_state=42))

print('\nДля ручной проверки откройте изображения по путям:')
print('top_crops/<filename>/...')
